 - Part 2: Model Training & Strategy Formulation

---

## 📋 Project Overview

**Objective:** Develop a predictive engine that translates features into actionable trading signals.


---

### Part 2 Goals:
1. **Data Preparation** - Load processed features and prepare train/validation/test splits
2. **Target Variable Definition** - Define what we're predicting (classification vs regression)
3. **Model Training** - Build and train multiple ML models
4. **Model Evaluation** - Evaluate model performance
5. **Signal Generation** - Convert predictions to trading signals
6. **Ensemble Strategy** - Combine multiple models for robustness

---

### Strategy Philosophy:

> *"Financial data is incredibly noisy. Relying on a single signal source or a single naive model architecture can often lead to instability."* - Task Hint

> *"Markets evolve. A relationship that held true in Year 1 may not exist in Year 3."* - Task Hint

We'll address these challenges by:
- Using an ensemble of diverse models
- Implementing walk-forward validation
- Focusing on robust signals rather than perfect predictions

---

## 📦 Step 1: Import Libraries

In [ ]:
# Install required packages (uncomment for Colab)
# !pip install pandas numpy matplotlib seaborn scikit-learn xgboost lightgbm catboost --quiet

# Core Libraries
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Sklearn
from sklearn.model_selection import TimeSeriesSplit, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.ensemble import AdaBoostClassifier, AdaBoostRegressor
from sklearn.ensemble import VotingClassifier, VotingRegressor
from sklearn.svm import SVC, SVR
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score,
    mean_squared_error, mean_absolute_error, r2_score
)
from sklearn.feature_selection import SelectFromModel, RFE
from sklearn.decomposition import PCA

# Gradient Boosting Libraries
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print("⚠️ XGBoost not available")

try:
    import lightgbm as lgb
    LGB_AVAILABLE = True
except ImportError:
    LGB_AVAILABLE = False
    print("⚠️ LightGBM not available")

# Statistical tests
from scipy import stats

print("✅ Libraries imported successfully!")
print(f"📅 Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 📥 Step 2: Load Processed Data

In [ ]:
# Load the processed data from Part 1
DATA_PATH = 'processed_features.csv'

try:
    df = pd.read_csv(DATA_PATH)
    print(f"✅ Data loaded successfully!")
    print(f"📊 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
except FileNotFoundError:
    print("❌ File not found. Please run Part 1 first or upload 'processed_features.csv'")

In [ ]:
# Display data info
print("📋 Data Overview:")
display(df.head())

print("\n📊 Data Types:")
print(df.dtypes)

In [ ]:
# Identify column names
# Adjust these based on your actual column names
ticker_col = None
date_col = None
close_col = None

for col in df.columns:
    col_lower = col.lower()
    if col_lower in ['ticker', 'symbol', 'stock', 'asset']:
        ticker_col = col
    elif col_lower in ['date', 'datetime', 'time']:
        date_col = col
    elif col_lower in ['close', 'adj_close', 'adjusted_close']:
        close_col = col

print(f"📌 Ticker column: {ticker_col}")
print(f"📌 Date column: {date_col}")
print(f"📌 Close column: {close_col}")

# Convert date column
if date_col:
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values([ticker_col, date_col]).reset_index(drop=True)
    
print(f"\n📅 Date range: {df[date_col].min()} to {df[date_col].max()}")

---

## 🎯 Step 3: Define Target Variable

We have two main approaches:

1. **Classification:** Predict if the stock goes UP, DOWN, or FLAT
2. **Regression:** Predict the actual forward return

We'll implement **both approaches** and combine them for a robust signal.

### Why both?
- Classification is more robust to noise but loses magnitude information
- Regression captures magnitude but is sensitive to outliers
- Combining them gives us the best of both worlds

In [ ]:
# Define prediction horizons
PREDICTION_HORIZONS = [1, 5, 10]  # Days ahead to predict

# Create forward returns for each horizon
for horizon in PREDICTION_HORIZONS:
    # Regression target: actual forward return
    df[f'target_return_{horizon}d'] = df.groupby(ticker_col)[close_col].pct_change(horizon).shift(-horizon)
    
    # Classification target: direction
    # 0 = Down (< -0.5%), 1 = Flat (-0.5% to 0.5%), 2 = Up (> 0.5%)
    df[f'target_class_{horizon}d'] = pd.cut(
        df[f'target_return_{horizon}d'],
        bins=[-np.inf, -0.005, 0.005, np.inf],
        labels=[0, 1, 2]  # Down, Flat, Up
    )
    
    # Binary classification: Up vs Not Up
    df[f'target_binary_{horizon}d'] = (df[f'target_return_{horizon}d'] > 0).astype(int)

print("✅ Target variables created!")
print(f"\nPrediction horizons: {PREDICTION_HORIZONS} days")

In [ ]:
# Analyze target distribution
print("📊 Target Variable Distribution:")
print("="*60)

for horizon in PREDICTION_HORIZONS:
    print(f"\n📈 {horizon}-Day Forward Return:")
    print(df[f'target_return_{horizon}d'].describe())
    
    print(f"\n📊 {horizon}-Day Direction (Binary):")
    print(df[f'target_binary_{horizon}d'].value_counts(normalize=True))

In [ ]:
# Visualize target distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, horizon in enumerate(PREDICTION_HORIZONS):
    axes[i].hist(df[f'target_return_{horizon}d'].dropna(), bins=50, edgecolor='black', alpha=0.7)
    axes[i].axvline(x=0, color='red', linestyle='--', linewidth=2)
    axes[i].set_title(f'{horizon}-Day Forward Return Distribution', fontweight='bold')
    axes[i].set_xlabel('Return')
    axes[i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

---

## 🔧 Step 4: Prepare Features and Train/Test Split

### Time Series Cross-Validation:

Unlike regular cross-validation, we must respect the temporal order of data:
- Training data must come before validation/test data
- We use **walk-forward validation** to simulate real trading conditions

In [ ]:
# Define feature columns (exclude identifiers and targets)
exclude_cols = [ticker_col, date_col]
exclude_cols += [col for col in df.columns if 'target' in col.lower()]
exclude_cols += [col for col in df.columns if 'forward' in col.lower()]

# Also exclude raw OHLCV columns (we want to use derived features)
ohlcv_patterns = ['open', 'high', 'low', 'close', 'volume', 'adj']
for col in df.columns:
    if any(pattern in col.lower() for pattern in ohlcv_patterns) and '_' not in col:
        if col not in exclude_cols:
            exclude_cols.append(col)

# Get feature columns
feature_cols = [col for col in df.columns if col not in exclude_cols]

print(f"📊 Feature Selection:")
print(f"   Total columns: {len(df.columns)}")
print(f"   Excluded columns: {len(exclude_cols)}")
print(f"   Feature columns: {len(feature_cols)}")
print(f"\n📋 Features: {feature_cols[:20]}..." if len(feature_cols) > 20 else f"\n📋 Features: {feature_cols}")

In [ ]:
# Time-based train/validation/test split
# We'll use chronological split to simulate real trading

# Remove rows with NaN targets
df_clean = df.dropna(subset=[f'target_return_{PREDICTION_HORIZONS[0]}d']).copy()

# Sort by date
df_clean = df_clean.sort_values(date_col).reset_index(drop=True)

# Get unique dates
unique_dates = df_clean[date_col].unique()
n_dates = len(unique_dates)

# Split: 60% train, 20% validation, 20% test
train_end_idx = int(n_dates * 0.6)
val_end_idx = int(n_dates * 0.8)

train_end_date = unique_dates[train_end_idx]
val_end_date = unique_dates[val_end_idx]

# Create splits
train_df = df_clean[df_clean[date_col] <= train_end_date].copy()
val_df = df_clean[(df_clean[date_col] > train_end_date) & (df_clean[date_col] <= val_end_date)].copy()
test_df = df_clean[df_clean[date_col] > val_end_date].copy()

print("📊 Train/Validation/Test Split:")
print("="*60)
print(f"\n📈 Training set:")
print(f"   Date range: {train_df[date_col].min()} to {train_df[date_col].max()}")
print(f"   Samples: {len(train_df):,}")

print(f"\n📊 Validation set:")
print(f"   Date range: {val_df[date_col].min()} to {val_df[date_col].max()}")
print(f"   Samples: {len(val_df):,}")

print(f"\n📉 Test set:")
print(f"   Date range: {test_df[date_col].min()} to {test_df[date_col].max()}")
print(f"   Samples: {len(test_df):,}")

In [ ]:
# Prepare feature matrices
X_train = train_df[feature_cols].copy()
X_val = val_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

# Primary target: 5-day forward return (good balance between noise and signal)
PRIMARY_HORIZON = 5

y_train_reg = train_df[f'target_return_{PRIMARY_HORIZON}d'].copy()
y_val_reg = val_df[f'target_return_{PRIMARY_HORIZON}d'].copy()
y_test_reg = test_df[f'target_return_{PRIMARY_HORIZON}d'].copy()

y_train_clf = train_df[f'target_binary_{PRIMARY_HORIZON}d'].copy()
y_val_clf = val_df[f'target_binary_{PRIMARY_HORIZON}d'].copy()
y_test_clf = test_df[f'target_binary_{PRIMARY_HORIZON}d'].copy()

print(f"📊 Dataset shapes:")
print(f"   X_train: {X_train.shape}")
print(f"   X_val: {X_val.shape}")
print(f"   X_test: {X_test.shape}")

In [ ]:
# Handle any remaining missing values and infinities
def clean_features(X):
    """Clean feature matrix by handling missing and infinite values."""
    X = X.replace([np.inf, -np.inf], np.nan)
    # Fill NaN with median
    for col in X.columns:
        if X[col].isna().any():
            X[col] = X[col].fillna(X[col].median())
    return X

X_train = clean_features(X_train)
X_val = clean_features(X_val)
X_test = clean_features(X_test)

print("✅ Features cleaned!")
print(f"   Remaining NaN in X_train: {X_train.isna().sum().sum()}")
print(f"   Remaining NaN in X_val: {X_val.isna().sum().sum()}")
print(f"   Remaining NaN in X_test: {X_test.isna().sum().sum()}")

In [ ]:
# Scale features using RobustScaler (less sensitive to outliers)
scaler = RobustScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=feature_cols,
    index=X_train.index
)

X_val_scaled = pd.DataFrame(
    scaler.transform(X_val),
    columns=feature_cols,
    index=X_val.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=feature_cols,
    index=X_test.index
)

print("✅ Features scaled using RobustScaler")

---

## 🤖 Step 5: Model Training

We'll train multiple models and combine them into an ensemble:

### Models to try:
1. **Logistic Regression** - Baseline linear model
2. **Random Forest** - Non-linear, handles feature interactions
3. **Gradient Boosting** - Sequential learning, often best performer
4. **XGBoost/LightGBM** - Optimized gradient boosting
5. **Neural Network** - Can capture complex patterns

### Why ensemble?
- Different models capture different patterns
- Reduces overfitting to any single approach
- More robust to market regime changes

In [ ]:
# Model Training - Classification
print("🤖 Training Classification Models...")
print("="*60)

# Store models and results
clf_models = {}
clf_results = {}

# 1. Logistic Regression
print("\n1️⃣ Training Logistic Regression...")
lr_clf = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr_clf.fit(X_train_scaled, y_train_clf)
clf_models['Logistic Regression'] = lr_clf
val_pred = lr_clf.predict(X_val_scaled)
clf_results['Logistic Regression'] = {
    'accuracy': accuracy_score(y_val_clf, val_pred),
    'f1': f1_score(y_val_clf, val_pred, average='weighted'),
    'roc_auc': roc_auc_score(y_val_clf, lr_clf.predict_proba(X_val_scaled)[:, 1])
}
print(f"   Validation Accuracy: {clf_results['Logistic Regression']['accuracy']:.4f}")
print(f"   Validation ROC-AUC: {clf_results['Logistic Regression']['roc_auc']:.4f}")

# 2. Random Forest
print("\n2️⃣ Training Random Forest...")
rf_clf = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_split=20,
    random_state=42, n_jobs=-1, class_weight='balanced'
)
rf_clf.fit(X_train_scaled, y_train_clf)
clf_models['Random Forest'] = rf_clf
val_pred = rf_clf.predict(X_val_scaled)
clf_results['Random Forest'] = {
    'accuracy': accuracy_score(y_val_clf, val_pred),
    'f1': f1_score(y_val_clf, val_pred, average='weighted'),
    'roc_auc': roc_auc_score(y_val_clf, rf_clf.predict_proba(X_val_scaled)[:, 1])
}
print(f"   Validation Accuracy: {clf_results['Random Forest']['accuracy']:.4f}")
print(f"   Validation ROC-AUC: {clf_results['Random Forest']['roc_auc']:.4f}")

# 3. Gradient Boosting
print("\n3️⃣ Training Gradient Boosting...")
gb_clf = GradientBoostingClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    subsample=0.8, random_state=42
)
gb_clf.fit(X_train_scaled, y_train_clf)
clf_models['Gradient Boosting'] = gb_clf
val_pred = gb_clf.predict(X_val_scaled)
clf_results['Gradient Boosting'] = {
    'accuracy': accuracy_score(y_val_clf, val_pred),
    'f1': f1_score(y_val_clf, val_pred, average='weighted'),
    'roc_auc': roc_auc_score(y_val_clf, gb_clf.predict_proba(X_val_scaled)[:, 1])
}
print(f"   Validation Accuracy: {clf_results['Gradient Boosting']['accuracy']:.4f}")
print(f"   Validation ROC-AUC: {clf_results['Gradient Boosting']['roc_auc']:.4f}")

In [ ]:
# 4. XGBoost (if available)
if XGB_AVAILABLE:
    print("\n4️⃣ Training XGBoost...")
    xgb_clf = xgb.XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        use_label_encoder=False, eval_metric='logloss'
    )
    xgb_clf.fit(X_train_scaled, y_train_clf)
    clf_models['XGBoost'] = xgb_clf
    val_pred = xgb_clf.predict(X_val_scaled)
    clf_results['XGBoost'] = {
        'accuracy': accuracy_score(y_val_clf, val_pred),
        'f1': f1_score(y_val_clf, val_pred, average='weighted'),
        'roc_auc': roc_auc_score(y_val_clf, xgb_clf.predict_proba(X_val_scaled)[:, 1])
    }
    print(f"   Validation Accuracy: {clf_results['XGBoost']['accuracy']:.4f}")
    print(f"   Validation ROC-AUC: {clf_results['XGBoost']['roc_auc']:.4f}")

# 5. LightGBM (if available)
if LGB_AVAILABLE:
    print("\n5️⃣ Training LightGBM...")
    lgb_clf = lgb.LGBMClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        verbose=-1
    )
    lgb_clf.fit(X_train_scaled, y_train_clf)
    clf_models['LightGBM'] = lgb_clf
    val_pred = lgb_clf.predict(X_val_scaled)
    clf_results['LightGBM'] = {
        'accuracy': accuracy_score(y_val_clf, val_pred),
        'f1': f1_score(y_val_clf, val_pred, average='weighted'),
        'roc_auc': roc_auc_score(y_val_clf, lgb_clf.predict_proba(X_val_scaled)[:, 1])
    }
    print(f"   Validation Accuracy: {clf_results['LightGBM']['accuracy']:.4f}")
    print(f"   Validation ROC-AUC: {clf_results['LightGBM']['roc_auc']:.4f}")

In [ ]:
# 6. Neural Network
print("\n6️⃣ Training Neural Network...")
nn_clf = MLPClassifier(
    hidden_layer_sizes=(100, 50, 25), activation='relu',
    max_iter=500, random_state=42, early_stopping=True,
    validation_fraction=0.1
)
nn_clf.fit(X_train_scaled, y_train_clf)
clf_models['Neural Network'] = nn_clf
val_pred = nn_clf.predict(X_val_scaled)
clf_results['Neural Network'] = {
    'accuracy': accuracy_score(y_val_clf, val_pred),
    'f1': f1_score(y_val_clf, val_pred, average='weighted'),
    'roc_auc': roc_auc_score(y_val_clf, nn_clf.predict_proba(X_val_scaled)[:, 1])
}
print(f"   Validation Accuracy: {clf_results['Neural Network']['accuracy']:.4f}")
print(f"   Validation ROC-AUC: {clf_results['Neural Network']['roc_auc']:.4f}")

print("\n✅ All classification models trained!")

In [ ]:
# Summary of classification results
print("📊 Classification Model Comparison:")
print("="*60)

results_df = pd.DataFrame(clf_results).T
results_df = results_df.sort_values('roc_auc', ascending=False)
display(results_df)

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
results_df['roc_auc'].plot(kind='barh', ax=ax, color='steelblue', edgecolor='black')
ax.set_xlabel('ROC-AUC Score')
ax.set_title('Classification Model Comparison (Validation Set)', fontweight='bold')
ax.axvline(x=0.5, color='red', linestyle='--', label='Random Baseline')
plt.legend()
plt.tight_layout()
plt.show()

---

## 🔮 Step 6: Regression Models (Predicting Actual Returns)

In [ ]:
# Model Training - Regression
print("🤖 Training Regression Models...")
print("="*60)

# Store models and results
reg_models = {}
reg_results = {}

# 1. Ridge Regression
print("\n1️⃣ Training Ridge Regression...")
ridge_reg = Ridge(alpha=1.0, random_state=42)
ridge_reg.fit(X_train_scaled, y_train_reg)
reg_models['Ridge'] = ridge_reg
val_pred = ridge_reg.predict(X_val_scaled)
reg_results['Ridge'] = {
    'rmse': np.sqrt(mean_squared_error(y_val_reg, val_pred)),
    'mae': mean_absolute_error(y_val_reg, val_pred),
    'r2': r2_score(y_val_reg, val_pred),
    'corr': np.corrcoef(y_val_reg, val_pred)[0, 1]
}
print(f"   Validation RMSE: {reg_results['Ridge']['rmse']:.6f}")
print(f"   Validation Corr: {reg_results['Ridge']['corr']:.4f}")

# 2. Random Forest Regressor
print("\n2️⃣ Training Random Forest Regressor...")
rf_reg = RandomForestRegressor(
    n_estimators=200, max_depth=10, min_samples_split=20,
    random_state=42, n_jobs=-1
)
rf_reg.fit(X_train_scaled, y_train_reg)
reg_models['Random Forest'] = rf_reg
val_pred = rf_reg.predict(X_val_scaled)
reg_results['Random Forest'] = {
    'rmse': np.sqrt(mean_squared_error(y_val_reg, val_pred)),
    'mae': mean_absolute_error(y_val_reg, val_pred),
    'r2': r2_score(y_val_reg, val_pred),
    'corr': np.corrcoef(y_val_reg, val_pred)[0, 1]
}
print(f"   Validation RMSE: {reg_results['Random Forest']['rmse']:.6f}")
print(f"   Validation Corr: {reg_results['Random Forest']['corr']:.4f}")

# 3. Gradient Boosting Regressor
print("\n3️⃣ Training Gradient Boosting Regressor...")
gb_reg = GradientBoostingRegressor(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    subsample=0.8, random_state=42
)
gb_reg.fit(X_train_scaled, y_train_reg)
reg_models['Gradient Boosting'] = gb_reg
val_pred = gb_reg.predict(X_val_scaled)
reg_results['Gradient Boosting'] = {
    'rmse': np.sqrt(mean_squared_error(y_val_reg, val_pred)),
    'mae': mean_absolute_error(y_val_reg, val_pred),
    'r2': r2_score(y_val_reg, val_pred),
    'corr': np.corrcoef(y_val_reg, val_pred)[0, 1]
}
print(f"   Validation RMSE: {reg_results['Gradient Boosting']['rmse']:.6f}")
print(f"   Validation Corr: {reg_results['Gradient Boosting']['corr']:.4f}")

In [ ]:
# Summary of regression results
print("📊 Regression Model Comparison:")
print("="*60)

reg_results_df = pd.DataFrame(reg_results).T
reg_results_df = reg_results_df.sort_values('corr', ascending=False)
display(reg_results_df)

# Note about R² in financial data
print("\n⚠️ Note: Low R² is expected in financial data due to high noise.")
print("   What matters more is the correlation between predictions and actual returns.")

---

## 🎯 Step 7: Feature Importance Analysis

In [ ]:
# Feature importance from Random Forest
print("📊 Feature Importance Analysis:")

# Get feature importances
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_clf.feature_importances_
}).sort_values('importance', ascending=False)

# Top 20 features
print("\n🏆 Top 20 Most Important Features:")
display(importance_df.head(20))

# Visualize
fig, ax = plt.subplots(figsize=(10, 8))
importance_df.head(20).plot(kind='barh', x='feature', y='importance', ax=ax, 
                             color='steelblue', edgecolor='black', legend=False)
ax.set_xlabel('Importance')
ax.set_title('Top 20 Features by Importance (Random Forest)', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

---

## 🎭 Step 8: Create Ensemble Model

We'll combine predictions from multiple models to create a more robust signal.

In [ ]:
# Create ensemble predictions
print("🎭 Creating Ensemble Model...")
print("="*60)

# Get probability predictions from all classification models
ensemble_probs = pd.DataFrame()

for name, model in clf_models.items():
    probs = model.predict_proba(X_val_scaled)[:, 1]  # Probability of going UP
    ensemble_probs[name] = probs

# Simple average ensemble
ensemble_probs['Ensemble_Avg'] = ensemble_probs.mean(axis=1)

# Weighted ensemble (based on validation ROC-AUC)
weights = {name: result['roc_auc'] for name, result in clf_results.items()}
total_weight = sum(weights.values())
weights = {k: v/total_weight for k, v in weights.items()}

ensemble_probs['Ensemble_Weighted'] = sum(
    ensemble_probs[name] * weights[name] for name in clf_models.keys()
)

print("Model weights (based on ROC-AUC):")
for name, weight in sorted(weights.items(), key=lambda x: x[1], reverse=True):
    print(f"   {name}: {weight:.3f}")

In [ ]:
# Evaluate ensemble performance
print("\n📊 Ensemble Performance:")
print("="*60)

for ensemble_type in ['Ensemble_Avg', 'Ensemble_Weighted']:
    pred_binary = (ensemble_probs[ensemble_type] > 0.5).astype(int)
    acc = accuracy_score(y_val_clf, pred_binary)
    roc = roc_auc_score(y_val_clf, ensemble_probs[ensemble_type])
    print(f"\n{ensemble_type}:")
    print(f"   Accuracy: {acc:.4f}")
    print(f"   ROC-AUC: {roc:.4f}")

---

## 📈 Step 9: Generate Trading Signals

Now we convert our predictions into actionable trading signals.

### Signal Generation Strategy:

1. **Probability Score:** Use ensemble probability as confidence
2. **Cross-Sectional Ranking:** Rank stocks by predicted returns
3. **Long-Short Portfolio:** Long top quintile, Short bottom quintile
4. **Signal Smoothing:** Avoid excessive turnover

In [ ]:
def generate_trading_signals(df_data, clf_models, reg_models, feature_cols, scaler, 
                             ticker_col, date_col, n_long=10, n_short=10):
    """
    Generate trading signals for each date.
    
    Parameters:
    -----------
    df_data : DataFrame with features
    clf_models : dict of classification models
    reg_models : dict of regression models  
    feature_cols : list of feature column names
    scaler : fitted scaler object
    ticker_col, date_col : column names
    n_long, n_short : number of stocks to long/short
    
    Returns:
    --------
    DataFrame with trading signals
    """
    
    signals_list = []
    
    for date in df_data[date_col].unique():
        # Get data for this date
        daily_data = df_data[df_data[date_col] == date].copy()
        
        if len(daily_data) < (n_long + n_short):
            continue
            
        # Prepare features
        X = daily_data[feature_cols].copy()
        X = X.replace([np.inf, -np.inf], np.nan)
        for col in X.columns:
            X[col] = X[col].fillna(X[col].median())
        
        X_scaled = pd.DataFrame(
            scaler.transform(X),
            columns=feature_cols,
            index=X.index
        )
        
        # Get ensemble probability (classification)
        probs = np.zeros(len(X_scaled))
        for name, model in clf_models.items():
            probs += model.predict_proba(X_scaled)[:, 1]
        probs /= len(clf_models)
        
        # Get predicted returns (regression)
        pred_returns = np.zeros(len(X_scaled))
        for name, model in reg_models.items():
            pred_returns += model.predict(X_scaled)
        pred_returns /= len(reg_models)
        
        # Combine signals: probability * predicted return magnitude
        combined_score = (probs - 0.5) * 2  # Scale to [-1, 1]
        combined_score = combined_score * np.abs(pred_returns) * 100  # Weight by magnitude
        
        # Create signal DataFrame
        signal_df = pd.DataFrame({
            ticker_col: daily_data[ticker_col].values,
            date_col: date,
            'prob_up': probs,
            'pred_return': pred_returns,
            'combined_score': combined_score
        })
        
        # Rank stocks
        signal_df['rank'] = signal_df['combined_score'].rank(ascending=False)
        
        # Generate position signals
        # Long top N, Short bottom N, Neutral rest
        signal_df['position'] = 0
        signal_df.loc[signal_df['rank'] <= n_long, 'position'] = 1  # Long
        signal_df.loc[signal_df['rank'] > len(signal_df) - n_short, 'position'] = -1  # Short
        
        signals_list.append(signal_df)
    
    return pd.concat(signals_list, ignore_index=True)

print("✅ Signal generation function defined!")

In [ ]:
# Generate signals for validation and test sets
print("📈 Generating Trading Signals...")
print("="*60)

# Validation signals
val_signals = generate_trading_signals(
    val_df, clf_models, reg_models, feature_cols, scaler,
    ticker_col, date_col, n_long=5, n_short=5
)
print(f"\n📊 Validation signals generated: {len(val_signals):,} rows")

# Test signals
test_signals = generate_trading_signals(
    test_df, clf_models, reg_models, feature_cols, scaler,
    ticker_col, date_col, n_long=5, n_short=5
)
print(f"📊 Test signals generated: {len(test_signals):,} rows")

# Display sample
print("\n📋 Sample Signals:")
display(test_signals.head(20))

In [ ]:
# Analyze signal distribution
print("📊 Signal Distribution Analysis:")
print("="*60)

print("\nPosition Distribution:")
print(test_signals['position'].value_counts())

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Probability distribution
axes[0].hist(test_signals['prob_up'], bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(x=0.5, color='red', linestyle='--', linewidth=2)
axes[0].set_title('Distribution of Up Probability', fontweight='bold')
axes[0].set_xlabel('Probability')
axes[0].set_ylabel('Frequency')

# Combined score distribution
axes[1].hist(test_signals['combined_score'], bins=50, edgecolor='black', alpha=0.7)
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[1].set_title('Distribution of Combined Score', fontweight='bold')
axes[1].set_xlabel('Score')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

---

## 💾 Step 10: Save Models and Signals

In [ ]:
import pickle

# Save models
models_dict = {
    'clf_models': clf_models,
    'reg_models': reg_models,
    'scaler': scaler,
    'feature_cols': feature_cols
}

with open('trained_models.pkl', 'wb') as f:
    pickle.dump(models_dict, f)

print("✅ Models saved to: trained_models.pkl")

# Save signals
test_signals.to_csv('trading_signals_test.csv', index=False)
val_signals.to_csv('trading_signals_val.csv', index=False)

print("✅ Trading signals saved!")

---

## 📝 Summary & Key Findings

### What We Accomplished:

1. **Target Definition:**
   - Created both classification (direction) and regression (return) targets
   - Primary horizon: 5-day forward returns

2. **Model Training:**
   - Trained 5+ classification models and 3+ regression models
   - Used time-based train/val/test split to avoid look-ahead bias
   - Applied proper feature scaling

3. **Ensemble Strategy:**
   - Combined multiple models using weighted averaging
   - Weights based on validation ROC-AUC performance

4. **Signal Generation:**
   - Created combined score using probability and predicted return magnitude
   - Implemented cross-sectional ranking for long-short portfolio

### Key Insights:

- Ensemble models typically outperform individual models
- Gradient boosting methods (XGBoost, LightGBM) often perform best
- Feature importance reveals which signals drive predictions

### Next Steps (Part 3):
- Implement backtesting framework
- Calculate performance metrics (Sharpe, Drawdown, etc.)
- Analyze strategy with and without transaction costs

---